In [9]:
import pandas as pd
from pydataxm import pydataxm
import datetime as dt


api_xm = pydataxm.ReadDB()

fecha_inicio = dt.date(2022, 1, 1)
fecha_fin = dt.date(2026, 8, 31)

# Demanda Comercial No Regulada 
print("Descargando Demanda No Regulada...")
df_demanda_nr = api_xm.request_data(
    "DemaComeNoReg",   
    "Sistema",         
    fecha_inicio, 
    fecha_fin
)

#Precio de Bolsa Nacional 
print("Descargando Precio de Bolsa Nacional...")
df_precio_bolsa = api_xm.request_data(
    "PrecBolsNaci",    
    "Sistema",         
    fecha_inicio, 
    fecha_fin
)

df_demanda_nr.to_csv('demanda_no_regulada_2022_2026.csv', index=False)
df_precio_bolsa.to_csv('precio_bolsa_2022_2026.csv', index=False)

print("Archivos CSV individuales exportados con éxito")

print("\n--- Vista previa de la Demanda No Regulada ---")
print(df_demanda_nr.head())

print("\n--- Vista previa del Precio de Bolsa ---")
print(df_precio_bolsa.head())

Descargando Demanda No Regulada...
Descargando Precio de Bolsa Nacional...
¡Archivos CSV individuales exportados con éxito!

--- Vista previa de la Demanda No Regulada ---
        Id Values_code  Values_Hour01  Values_Hour02  Values_Hour03  \
0  Sistema     Sistema  1908294.77000  1898997.50000  1898374.94000   
1  Sistema     Sistema  1971438.46000  1918030.54000  1947298.67000   
2  Sistema     Sistema  2262048.84000  2269005.56000  2278488.79000   
3  Sistema     Sistema  2581873.49000  2578290.04000  2524672.62000   
4  Sistema     Sistema  2625907.33000  2583675.42000  2602741.29000   

   Values_Hour04  Values_Hour05  Values_Hour06  Values_Hour07  Values_Hour08  \
0  1879004.44000  1881200.96000  1889830.83000  1612076.29000  1595284.92000   
1  1950661.39000  1938981.73000  1977079.03000  1733384.75000  1773140.04000   
2  2278675.67000  2304167.92000  2344791.76000  2209725.78000  2348181.32000   
3  2524137.42000  2516856.34000  2564719.94000  2396038.57000  2520630.58000   
4

In [8]:
import pandas as pd
from pydataxm import pydataxm
import datetime as dt

api_xm = pydataxm.ReadDB()
anios = [2022, 2023, 2024, 2025, 2026]

variables = {
    "AporEner": "aportes_hidricos_2022_2026.csv",
    "VoluUtilDiarEner": "volumen_util_2022_2026.csv",
    "PrecPromContNoRegu": "precios_contratos_2022_2026.csv"
}

for codigo_var, nombre_archivo in variables.items():
    print(f"\nIniciando descarga de la métrica: {codigo_var}")
    df_consolidado = pd.DataFrame()
    
    for anio in anios:
        print(f" -> Consultando bloque del año {anio}...")
        
        
        fecha_inicio = dt.date(anio, 1, 1)
        if anio == 2026:
            fecha_fin = dt.date(2026, 8, 31)
        else:
            fecha_fin = dt.date(anio, 12, 31)
            
        
        try:
            df_temp = api_xm.request_data(codigo_var, "Sistema", fecha_inicio, fecha_fin)
            df_consolidado = pd.concat([df_consolidado, df_temp], ignore_index=True)
        except Exception as e:
            print(f" [!] Error al descargar {anio} para {codigo_var}: {e}")
            
    if not df_consolidado.empty:
        df_consolidado.to_csv(nombre_archivo, index=False)
        print(f"Exportación exitosa: {nombre_archivo} guardado en disco")


Iniciando descarga de la métrica: AporEner
 -> Consultando bloque del año 2022...
 -> Consultando bloque del año 2023...
 -> Consultando bloque del año 2024...
 -> Consultando bloque del año 2025...
 -> Consultando bloque del año 2026...
¡Exportación exitosa: aportes_hidricos_2022_2026.csv guardado en el disco!

Iniciando descarga de la métrica: VoluUtilDiarEner
 -> Consultando bloque del año 2022...
 -> Consultando bloque del año 2023...
 -> Consultando bloque del año 2024...
 -> Consultando bloque del año 2025...
 -> Consultando bloque del año 2026...
¡Exportación exitosa: volumen_util_2022_2026.csv guardado en el disco!

Iniciando descarga de la métrica: PrecPromContNoRegu
 -> Consultando bloque del año 2022...
 -> Consultando bloque del año 2023...
 -> Consultando bloque del año 2024...
 -> Consultando bloque del año 2025...
 -> Consultando bloque del año 2026...
¡Exportación exitosa: precios_contratos_2022_2026.csv guardado en el disco!


In [19]:
import pandas as pd

def aplanar_matriz_xm(df, nombre_variable):
    df_melt = pd.melt(
        df, 
        id_vars=['Date'], 
        value_vars=[col for col in df.columns if 'Values_Hour' in col],
        var_name='Hora_String', 
        value_name=nombre_variable
    )
    df_melt['Hora_Int'] = df_melt['Hora_String'].str.extract(r'(\d+)').astype(int) - 1
    df_melt['FechaHora'] = pd.to_datetime(df_melt['Date']) + pd.to_timedelta(df_melt['Hora_Int'], unit='h')
    return df_melt[['FechaHora', 'Date', nombre_variable]].sort_values('FechaHora').reset_index(drop=True)

df_contratos = pd.read_csv('precios_contratos_2022_2026.csv')

df_master = pd.merge(
    aplanar_matriz_xm(df_demanda_nr, 'Demanda_No_Regulada_kWh'),
    aplanar_matriz_xm(df_precio_bolsa, 'Precio_Bolsa_COP').drop(columns=['Date']),
    on='FechaHora', how='inner'
)
df_master = pd.merge(
    df_master,
    aplanar_matriz_xm(df_contratos, 'Precio_Contrato_COP').drop(columns=['Date']),
    on='FechaHora', how='inner'
)

df_aportes = pd.read_csv('aportes_hidricos_2022_2026.csv').rename(columns={'Value': 'Aportes_Hidricos_kWh'})
df_volumen = pd.read_csv('volumen_util_2022_2026.csv').rename(columns={'Value': 'Volumen_Util_kWh'})

df_master['Date'] = pd.to_datetime(df_master['Date'])
df_aportes['Date'] = pd.to_datetime(df_aportes['Date'])
df_volumen['Date'] = pd.to_datetime(df_volumen['Date'])

df_master = pd.merge(df_master, df_aportes[['Date', 'Aportes_Hidricos_kWh']], on='Date', how='left')
df_master = pd.merge(df_master, df_volumen[['Date', 'Volumen_Util_kWh']], on='Date', how='left')

df_master['Demanda_No_Regulada_kWh'] = pd.to_numeric(df_master['Demanda_No_Regulada_kWh'], errors='coerce')
df_master['Precio_Bolsa_COP'] = pd.to_numeric(df_master['Precio_Bolsa_COP'], errors='coerce')

df_master = df_master.drop(columns=['Date'])
df_master.set_index('FechaHora', inplace=True)

df_master.to_parquet('dataset_master_optimizado.parquet', engine='fastparquet')

print("Estandarización de bases de datos completada")

Estandarización de bases de datos completada


In [21]:
print(f"Tamaño de la matriz (Filas, Columnas): {df_master.shape}")
print(f"Total de datos extraídos: {df_master.size}")
print("\n--- Información detallada de la matriz ---")
df_master.info()
df_master.head(10)

Tamaño de la matriz (Filas, Columnas): (40896, 5)
Total de datos extraídos: 204480

--- Información detallada de la matriz ---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 40896 entries, 2022-01-01 00:00:00 to 2026-08-31 23:00:00
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Demanda_No_Regulada_kWh  40896 non-null  float64
 1   Precio_Bolsa_COP         40896 non-null  float64
 2   Precio_Contrato_COP      40896 non-null  float64
 3   Aportes_Hidricos_kWh     40896 non-null  float64
 4   Volumen_Util_kWh         40896 non-null  float64
dtypes: float64(5)
memory usage: 1.9 MB


,Demanda_No_Regulada_kWh,Precio_Bolsa_COP,Precio_Contrato_COP,Aportes_Hidricos_kWh,Volumen_Util_kWh
FechaHora,,,,,
2022-01-01 00:00:00,1908294.77,186.57681,245.52680,109542700.0,1.331105e+10
2022-01-01 01:00:00,1898997.50,186.57681,245.47060,109542700.0,1.331105e+10
2022-01-01 02:00:00,1898374.94,186.57681,245.28809,109542700.0,1.331105e+10
2022-01-01 03:00:00,1879004.44,198.25581,245.38345,109542700.0,1.331105e+10
2022-01-01 04:00:00,1881200.96,186.57681,245.15850,109542700.0,1.331105e+10
2022-01-01 05:00:00,1889830.83,186.57681,245.22537,109542700.0,1.331105e+10
2022-01-01 06:00:00,1612076.29,185.57681,245.30950,109542700.0,1.331105e+10
2022-01-01 07:00:00,1595284.92,183.55681,244.77560,109542700.0,1.331105e+10
2022-01-01 08:00:00,1613834.34,185.57681,244.36670,109542700.0,1.331105e+10


In [ ]:
df_master.to_csv('dataset_master_optimizado.csv')